# H&M Product Recommendation Engine
**Author:** Karim Mattar | DS207 Applied Machine Learning, Spring 2026

## Overview
2-stage recommendation system:
1. **KNN Retrieval** — narrows 105k products to ~100 candidates per customer using cosine similarity on product feature vectors
2. **Ranker** — scores each candidate with a Neural Net and a Random Forest, picks top 12

**Evaluation:** Precision@12, Recall@12, F1@12 (per customer, then averaged)

In [ ]:
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras

sys.path.insert(0, '../src')
from customer_features import CustomerFeatureEngineer
from product_features import ProductFeatureEngineer
from recommendation_training import RecommendationTrainingBuilder

warnings.filterwarnings('ignore')
np.random.seed(67)
tf.random.set_seed(67)

In [ ]:
# Raw cleaned CSVs
DATA_DIR = Path('/Users/karimmattar11/Desktop/Berkeley/ds207/Project/archive (4)')

articles = pd.read_csv(DATA_DIR / 'articles_hm_cleaned.csv')
customers = pd.read_csv(DATA_DIR / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions_hm_cleaned.csv')
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

print(f"Articles: {articles.shape}")
print(f"Customers: {customers.shape}")
print(f"Transactions: {transactions.shape}")
print(f"Date range: {transactions['t_dat'].min()} → {transactions['t_dat'].max()}")

In [ ]:
PKL_DIR = Path('../data/processed/product_recommendation')

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

X_train = load_pkl(PKL_DIR / 'X_train_base.pkl')
y_train = load_pkl(PKL_DIR / 'y_train_base.pkl')
X_val   = load_pkl(PKL_DIR / 'X_val_base.pkl')
y_val   = load_pkl(PKL_DIR / 'y_val_base.pkl')
X_test  = load_pkl(PKL_DIR / 'X_test_base.pkl')
y_test  = load_pkl(PKL_DIR / 'y_test_base.pkl')

print(f"Train: {X_train.shape}, positives: {y_train.sum()}")
print(f"Val:   {X_val.shape},   positives: {y_val.sum()}")
print(f"Test:  {X_test.shape},  positives: {y_test.sum()}")
print(f"Features ({X_train.shape[1]}): {X_train.columns.tolist()}")

## 1. Recommendation-Specific EDA

Before building the model, we analyze three properties of the data that directly inform design choices:
1. **Product popularity distribution** — justifies KNN retrieval (most products are rarely bought)
2. **Cold-start customer share** — shows how many customers have no purchase history per split
3. **Category distribution** — shows which garment groups dominate purchases

In [ ]:
# Temporal split cutoffs (same as all other notebooks)
TRAIN_CUTOFF = pd.Timestamp('2019-09-30')
VAL_CUTOFF   = pd.Timestamp('2019-10-31')
TEST_CUTOFF  = pd.Timestamp('2019-11-30')

# Transactions per product
product_counts = transactions.groupby('article_id').size().sort_values(ascending=False).reset_index()
product_counts.columns = ['article_id', 'num_transactions']

active_products = product_counts.shape[0]
total_products = articles.shape[0]
top_10pct_share = product_counts.head(int(active_products * 0.1))['num_transactions'].sum() / product_counts['num_transactions'].sum() * 100
print(f"Active products: {active_products:,} / {total_products:,} ({active_products/total_products*100:.1f}%)")
print(f"Top 10% of products account for {top_10pct_share:.1f}% of all transactions")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(product_counts['num_transactions'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_xlabel('Transactions per Product')
axes[0].set_ylabel('Number of Products (log scale)')
axes[0].set_title('Product Popularity Distribution (Log Scale)')

product_counts['cumulative_pct'] = product_counts['num_transactions'].cumsum() / product_counts['num_transactions'].sum() * 100
axes[1].plot(range(len(product_counts)), product_counts['cumulative_pct'], color='steelblue')
axes[1].axhline(y=80, color='red', linestyle='--', label='80% of transactions')
axes[1].set_xlabel('Product Rank (most → least popular)')
axes[1].set_ylabel('Cumulative % of Transactions')
axes[1].set_title('Cumulative Transactions by Product Rank')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_product_popularity.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
def cold_start_share(transactions, cutoff):
    customers_with_history = transactions[transactions['t_dat'] <= cutoff]['customer_id'].unique()
    total = transactions['customer_id'].nunique()
    warm = len(customers_with_history)
    cold = total - warm
    return warm, cold, cold / total * 100

print("Cold-start analysis per temporal split:")
for name, cutoff in [('Train (as-of Sep 30)', TRAIN_CUTOFF),
                     ('Val   (as-of Oct 31)', VAL_CUTOFF),
                     ('Test  (as-of Nov 30)', TEST_CUTOFF)]:
    warm, cold, pct = cold_start_share(transactions, cutoff)
    print(f"  {name}: warm={warm:,}, cold-start={cold:,} ({pct:.1f}%)")

In [ ]:
txn_with_garment = transactions.merge(
    articles[['article_id', 'garment_group_name']],
    on='article_id', how='left'
)
top_garments = txn_with_garment['garment_group_name'].value_counts().head(10)

plt.figure(figsize=(10, 4))
plt.barh(top_garments.index[::-1], top_garments.values[::-1], color='steelblue')
plt.xlabel('Number of Transactions')
plt.title('Top 10 Garment Groups by Purchase Volume')
plt.tight_layout()
plt.savefig('eda_garment_categories.png', dpi=100, bbox_inches='tight')
plt.show()

**EDA Findings:**
- **Long tail:** The top 10% of products account for the majority of all transactions. Most of the 105k products are rarely purchased — KNN retrieval is essential to avoid scoring all 105k products per customer.
- **Cold-start:** A meaningful share of customers have no purchase history before each cutoff. These are handled by assigning them the mean product vector across all active customers.
- **Category skew:** Jersey Basic and Trousers dominate purchases. Our garment-group one-hot features capture this signal for the ranker.

## 2. Stage 1 — KNN Retrieval

For each customer, we build a "product interest vector" by averaging the 8 product feature vectors of all products they purchased in the training window. We use cosine-similarity KNN to retrieve the 100 most similar products from all ~51k active products. Cold-start customers (no purchase history) receive the mean vector across all active customers.

In [ ]:
PRODUCT_FEATURE_COLS = [
    'sales_last_7_days', 'sales_last_30_days',
    'days_since_first_sale', 'days_since_last_sale',
    'avg_price', 'min_price', 'max_price', 'product_price_std'
]

pfe_train = ProductFeatureEngineer(articles, transactions)
product_features_train = pfe_train.calculate_all_features(as_of_date=TRAIN_CUTOFF)

print(f"Active products with features: {product_features_train.shape[0]:,}")
print(product_features_train[PRODUCT_FEATURE_COLS].describe().round(4))

In [ ]:
def build_customer_product_vectors(transactions, product_features, as_of_date, feature_cols):
    """
    For each customer, average the product feature vectors of all products
    they purchased on or before as_of_date.
    Cold-start customers get the mean vector across all active customers.
    Returns DataFrame with customer_id + len(feature_cols) columns.
    """
    history = transactions[transactions['t_dat'] <= as_of_date]
    
    history_with_features = history.merge(
        product_features[['article_id'] + feature_cols],
        on='article_id', how='inner'
    )
    
    customer_vectors = history_with_features.groupby('customer_id')[feature_cols].mean().reset_index()
    
    mean_vector = product_features[feature_cols].mean()
    all_customers = transactions['customer_id'].unique()
    warm_customers = set(customer_vectors['customer_id'])
    cold_customers = [c for c in all_customers if c not in warm_customers]
    
    if cold_customers:
        cold_df = pd.DataFrame({'customer_id': cold_customers})
        for col in feature_cols:
            cold_df[col] = mean_vector[col]
        customer_vectors = pd.concat([customer_vectors, cold_df], ignore_index=True)
    
    print(f"Warm customers: {len(warm_customers):,}")
    print(f"Cold-start customers (assigned mean vector): {len(cold_customers):,}")
    
    return customer_vectors

customer_vectors_train = build_customer_product_vectors(
    transactions, product_features_train, TRAIN_CUTOFF, PRODUCT_FEATURE_COLS
)
print(f"Customer vectors shape: {customer_vectors_train.shape}")

In [ ]:
product_matrix = product_features_train[PRODUCT_FEATURE_COLS].values

knn = NearestNeighbors(n_neighbors=100, metric='cosine', algorithm='brute', n_jobs=-1)
knn.fit(product_matrix)

print(f"KNN fitted on {product_matrix.shape[0]:,} products with {product_matrix.shape[1]} features")

In [ ]:
def retrieve_candidates(customer_vectors, product_features, knn, feature_cols, sample_n=None):
    """
    For each customer, retrieve top 100 candidate article_ids via KNN.
    Returns DataFrame with columns: customer_id, article_id
    """
    article_ids = product_features['article_id'].values
    
    if sample_n:
        customer_vectors = customer_vectors.sample(n=sample_n, random_state=67).reset_index(drop=True)
    
    query_matrix = customer_vectors[feature_cols].values
    _, indices = knn.kneighbors(query_matrix)
    
    rows = []
    for i, customer_id in enumerate(customer_vectors['customer_id']):
        for article_id in article_ids[indices[i]]:
            rows.append({'customer_id': customer_id, 'article_id': article_id})
    
    return pd.DataFrame(rows)

EVAL_SAMPLE_N = 5000
candidates_train = retrieve_candidates(
    customer_vectors_train, product_features_train, knn, PRODUCT_FEATURE_COLS, sample_n=EVAL_SAMPLE_N
)
print(f"Candidates generated: {len(candidates_train):,} ({EVAL_SAMPLE_N} customers × 100 products)")